# Setup

In [1]:
#load basic libraries
import polars as pl  # It is advised to use polars, as the detasets can be quite memory-intensive
import numpy as np
import torch.nn as nn
from torch import optim
import torch
import re
from tqdm import trange
import random
from utils import *
from data.simulate_walk_the_book import *
import warnings



%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
dataset = "BTCUSDT"
data_root = "data"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_direct = f"{data_root}/{dataset}"
X_train = pl.read_parquet(f"{data_direct}/X_train.parquet")
y_train = pl.read_parquet(f"{data_direct}/y_train.parquet")

y_train_raw = y_train.__copy__()

# load the volume to fill info from the text file
with open(f"{data_direct}/vol_to_fill.txt", "r") as file:
    content = file.read().strip()

match = re.search(r"The volume to fill is: ([\d.]+)", content)
if match:
    volume_to_fill  = float(match.group(1))
    print(f"Extracted number: {volume_to_fill}")
else:
    print("No number found in the file.")


Extracted number: 4.0


In [3]:
X_train, y_train, means, stds, X_id_map, y_id_map = preprocess(X_train, y_train, device)

# id_map contains the mapping from index along the second dimension to anonymized_id used in the tensor
# X_train, y_train have shape (seq_len, num_ids, num_features)

# TWAP

In [ ]:
# Get ID values
id_values = X_id_map[:, 1].cpu().numpy()

# id column
ids_repeated = np.repeat(id_values, 60)

# time_in_hour column
times_seconds = np.arange(59*60, 59*60 + 60) * 1000
unique_ids = X_train.shape[1]
times_repeated = np.tile(times_seconds, unique_ids)

twap_preds = pl.DataFrame({
    'anonymized_id': ids_repeated,
    'time_in_hour': times_repeated,
    'position': volume_to_fill / 60 # Ensure volume is flat
}).with_columns(
    # This cast turns raw integers into the Duration type that prints as '59m 1s'
    pl.col('time_in_hour').cast(pl.Duration(time_unit='ms'))
)

twap_preds

# last k seconds

# Computing Basis Points

When walking the book, we use unmodified y_train.

> bps scores are different than in the example as we're dropping some anonymized ids during data cleaning.

In [ ]:
compute_bps(twap_preds, y_train_raw, volume_to_fill)

# Pipeline (accumulate this as you go)

Here, we define a pipeline where we create all relevant metrics across all datasets.

we define the following fields
- data asset
- model
- basis points

In [ ]:
warnings.filterwarnings("ignore")

model_type = "TWAP"
data_root = "data"

# 1. Initialize an empty list to collect results
results_accumulator = []

for dataset in DATASETS:
    # (Your existing loading logic)
    data_direct = f"{data_root}/{dataset}" # Ensuring path is correct
    X_train = pl.read_parquet(f"{data_direct}/X_train.parquet")
    y_train = pl.read_parquet(f"{data_direct}/y_train.parquet")
    y_train_raw = y_train.__copy__()

    with open(f"{data_direct}/vol_to_fill.txt", "r") as file:
        content = file.read().strip()
    match = re.search(r"The volume to fill is: ([\d.]+)", content)
    volume_to_fill = float(match.group(1))

    X_train, y_train, means, stds, X_id_map, y_id_map = preprocess(X_train, y_train, device)

    if model_type == "TWAP":
        id_values = X_id_map[:, 1].cpu().numpy()
        ids_repeated = np.repeat(id_values, 60)
        
        # Note: I'm using 60 here to match your np.repeat(..., 60)
        times_seconds = np.arange(59*60, 59*60 + 60) * 1000
        unique_ids = X_train.shape[1]
        times_repeated = np.tile(times_seconds, unique_ids)

        twap_preds = pl.DataFrame({
            'anonymized_id': ids_repeated,
            'time_in_hour': times_repeated,
            'position': volume_to_fill / 60 
        }).with_columns(
            pl.col('time_in_hour').cast(pl.Duration(time_unit='ms'))
        )

        # 2. Capture the result from your compute function
        # (Assuming compute_bps returns the mean float value)
        bps_score = compute_bps(twap_preds, y_train_raw, volume_to_fill)

        # 3. Append to the accumulator
        results_accumulator.append({
            "data asset": dataset,
            "model": model_type,
            "basis points": bps_score
        })

# 4. Create the final DataFrame
model_eval_df = pl.DataFrame(results_accumulator)

# Display results
model_eval_df